In [ ]:
import Pkg
Pkg.add("DynamicPolynomials")
Pkg.add("MultivariatePolynomials")
Pkg.add("SumOfSquares")
Pkg.add("CSDP")
Pkg.add("JuMP")
Pkg.add("SCS")
Pkg.add("PyPlot")
Pkg.build("PyPlot")


    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed DynamicPolynomials ────── v0.6.8
   Installed StarAlgebras ──────────── v0.3.0
   Installed MultivariatePolynomials ─ v0.5.20
   Installed MutableArithmetics ────── v1.8.0
    Updating `~/.julia/environments/v1.12/Project.toml`
  [7c1d4256] + DynamicPolynomials v0.6.8
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [7c1d4256] + DynamicPolynomials v0.6.8
  [102ac46a] + MultivariatePolynomials v0.5.20
  [d8a4904e] + MutableArithmetics v1.8.0
  [0c0c59c1] + StarAlgebras v0.3.0
Precompiling packages...
  17410.1 ms  ✓ MutableArithmetics
   5931.3 ms  ✓ StarAlgebras
   6263.2 ms  ✓ MultivariatePolynomials
   6666.3 ms  ✓ MultivariatePolynomials → LatexifyExt
   8983.1 ms  ✓ MultivariatePolynomials → MultivariatePolynomialsChainRulesCoreExt
  11582.1 ms  ✓ DynamicPolynomials
  6 dependencies successfully precompiled in 50 seconds. 491 already precompiled.
   Resolving package 

In [ ]:
# ============================================================
# Sum-of-Squares certificate for exponential stability
# ============================================================
#
# System:
#
#       ẋ₁ = -x₁ - x₁³
#       ẋ₂ = -2x₂ - x₂³
#
# We search for a quadratic Lyapunov function
#
#       V(x) = x' P x
#
# satisfying
#
#       V(x) >= eps * ||x||²
#
# and
#
#       V̇(x) <= -alpha * V(x)
#
# globally.
#
# Both conditions are imposed as SOS constraints.
#
# ============================================================


# ------------------------------------------------------------
# 1. Install packages in Google Colab
# ------------------------------------------------------------


# ------------------------------------------------------------
# 2. Load packages
# ------------------------------------------------------------

using JuMP
using SumOfSquares
using DynamicPolynomials
using MultivariatePolynomials
using LinearAlgebra
using CSDP


# ------------------------------------------------------------
# 3. Define the polynomial system
# ------------------------------------------------------------

@polyvar x[1:2]

# ẋ = f(x)
f = [
    -x[1] - x[1]^3,
    -2*x[2] - x[2]^3
]


# ------------------------------------------------------------
# 4. Parameters
# ------------------------------------------------------------

# Small positive number used to enforce positive definiteness:
#
#       V(x) >= eps ||x||²
#
# This is equivalent to P >= eps I for a quadratic V.
eps = 1e-3

# Binary-search interval for alpha.
#
# For this example alpha = 2 is the theoretical upper limit,
# because the slowest linear mode has eigenvalue -1.
#
# We deliberately use an upper bound slightly above 2.
alpha_low  = 0.0
alpha_high = 2.2

# Number of binary-search iterations.
#
# Each iteration solves one SDP.
n_iterations = 15


# ------------------------------------------------------------
# 5. Function that checks SOS feasibility for a given alpha
# ------------------------------------------------------------

function check_alpha(alpha, x, f, eps)

    n = length(x)

    # CSDP is an SDP solver used by SumOfSquares.jl.
    solver = optimizer_with_attributes(
        CSDP.Optimizer,
        MOI.Silent() => true
    )

    model = SOSModel(solver)

    # --------------------------------------------------------
    # Decision variable P
    #
    # P is constrained to be positive semidefinite.
    # --------------------------------------------------------

    @variable(model, P[1:n, 1:n], PSD)

    # Fix the scale of P.
    #
    # Without this, P can be multiplied by an arbitrary
    # positive scalar. The trace constraint removes this
    # scaling ambiguity.
    @constraint(model, sum(P[i,i] for i=1:n) == 1.0)


    # --------------------------------------------------------
    # Construct V(x) = x' P x
    # --------------------------------------------------------

    V = sum(
        P[i,j] * x[i] * x[j]
        for i=1:n, j=1:n
    )


    # --------------------------------------------------------
    # Compute V̇ = ∇V(x)' f(x)
    # --------------------------------------------------------

    gradV = differentiate.(V, x)

    Vdot = sum(
        gradV[i] * f[i]
        for i=1:n
    )


    # --------------------------------------------------------
    # SOS condition 1:
    #
    #       V(x) - eps ||x||² >= 0
    #
    # This guarantees
    #
    #       V(x) >= eps ||x||²
    #
    # and hence P is positive definite.
    # --------------------------------------------------------

    norm_squared = sum(x[i]^2 for i=1:n)

    @constraint(
        model,
        V - eps * norm_squared >= 0
    )


    # --------------------------------------------------------
    # SOS condition 2:
    #
    #       -V̇(x) - alpha V(x) >= 0
    #
    # equivalently
    #
    #       V̇(x) <= -alpha V(x)
    # --------------------------------------------------------

    @constraint(
        model,
        -Vdot - alpha * V >= 0
    )


    # --------------------------------------------------------
    # Solve the SDP
    # --------------------------------------------------------

    optimize!(model)

    status = termination_status(model)

    feasible = (
        status == MOI.OPTIMAL ||
        status == MOI.LOCALLY_SOLVED
    )

    return feasible
end


# ------------------------------------------------------------
# 6. Search for the largest feasible alpha
# ------------------------------------------------------------
#
# Why binary search?
#
# If the SOS conditions are feasible for some alpha,
# they remain feasible for every smaller alpha:
#
#       V̇ <= -alpha V
#
# automatically implies
#
#       V̇ <= -alpha_small V
#
# whenever alpha_small < alpha.
#
# Therefore feasibility is monotone in alpha.
#
# We can exploit this by repeatedly testing the midpoint:
#
#       alpha_mid = (alpha_low + alpha_high)/2
#
# If it is feasible, move the lower bound upward.
# If it is infeasible, move the upper bound downward.
#
# After several iterations, alpha_low is a good approximation
# of the largest certifiable alpha.
# ------------------------------------------------------------

println("Searching for the largest SOS-certified alpha...")

for k in 1:n_iterations

    alpha_mid = (alpha_low + alpha_high) / 2

    feasible = check_alpha(
        alpha_mid,
        x,
        f,
        eps
    )

    if feasible
        # alpha_mid can be certified, so try a larger alpha.
        alpha_low = alpha_mid

        println(
            "iteration $(k): alpha = ",
            round(alpha_mid, digits=8),
            "  -> feasible"
        )

    else
        # alpha_mid cannot currently be certified,
        # so search below it.
        alpha_high = alpha_mid

        println(
            "iteration $(k): alpha = ",
            round(alpha_mid, digits=8),
            "  -> infeasible"
        )
    end
end


# ------------------------------------------------------------
# 7. Final result
# ------------------------------------------------------------

println()
println("==============================================")
println("Largest certified alpha ≈ ", alpha_low)
println("==============================================")


# ------------------------------------------------------------
# 8. Optional: solve once more at the final alpha
#    and display the resulting P
# ------------------------------------------------------------

function solve_final(alpha, x, f, eps)

    n = length(x)

    solver = optimizer_with_attributes(
        CSDP.Optimizer,
        MOI.Silent() => true
    )

    model = SOSModel(solver)

    @variable(model, P[1:n,1:n], PSD)

    @constraint(
        model,
        sum(P[i,i] for i=1:n) == 1.0
    )

    V = sum(
        P[i,j] * x[i] * x[j]
        for i=1:n, j=1:n
    )

    gradV = differentiate.(V, x)

    Vdot = sum(
        gradV[i] * f[i]
        for i=1:n
    )

    norm_squared = sum(x[i]^2 for i=1:n)

    @constraint(
        model,
        V - eps * norm_squared >= 0
    )

    @constraint(
        model,
        -Vdot - alpha * V >= 0
    )

    optimize!(model)

    return model, P, V, Vdot
end


model, P, V, Vdot = solve_final(
    alpha_low,
    x,
    f,
    eps
)

println()
println("Certified Lyapunov matrix P:")
println(value.(P))

println()
println("V(x) = ")
println(value(V))

Searching for the largest SOS-certified alpha...
iteration 1: alpha = 1.1  -> feasible
iteration 2: alpha = 1.65  -> feasible
iteration 3: alpha = 1.925  -> feasible
iteration 4: alpha = 2.0625  -> infeasible
iteration 5: alpha = 1.99375  -> feasible
iteration 6: alpha = 2.028125  -> infeasible
iteration 7: alpha = 2.0109375  -> infeasible
iteration 8: alpha = 2.00234375  -> infeasible
iteration 9: alpha = 1.99804688  -> feasible
iteration 10: alpha = 2.00019531  -> infeasible
iteration 11: alpha = 1.99912109  -> feasible
iteration 12: alpha = 1.9996582  -> feasible
iteration 13: alpha = 1.99992676  -> feasible
iteration 14: alpha = 2.00006104  -> infeasible
iteration 15: alpha = 1.9999939  -> feasible

Largest certified alpha ≈ 1.9999938964843749

Certified Lyapunov matrix P:
[0.43752333930016524 0.0; 0.0 0.5624766606998358]

V(x) = 
0.5624766606998358*x[2]^2 + 0.43752333930016524*x[1]^2
